In [1]:
import pickle
from collections import OrderedDict
import os
import torch
import numpy as np
import pandas as pd
import sklearn
import random
import h5py
from sklearn.model_selection import GroupKFold
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import scanpy as sc
import anndata as ad
from textwrap import shorten

In [2]:
h5_src = "./Code/other_models/TranSiGen/data/Meisheng_used_data/processed_data.h5"

with h5py.File(h5_src, "r") as f:
    smiles_raw = np.asarray(f["canonical_smiles"])   # bytes or str
    cid_raw    = np.asarray(f["cid"])                # bytes
    sig_raw    = np.asarray(f["sig"])                # bytes
    n_samples  = smiles_raw.shape[0]

print("Samples:", n_samples)

Samples: 836649


In [3]:
pkl_idx2smi = (
    "./Code/other_models/TranSiGen/data/Meisheng_used_data/"
    "idx2smi.pickle"
)

with open(pkl_idx2smi, "rb") as fh:
    idx2smi = pickle.load(fh)          # {int: str}

smi2idx = {smi: idx for idx, smi in idx2smi.items()}

In [4]:
# convert bytes → str if needed
if isinstance(smiles_raw[0], (bytes, np.bytes_)):
    smiles_str = smiles_raw.astype(str)
else:
    smiles_str = smiles_raw

canon_idx = np.array([smi2idx[s] for s in smiles_str], dtype=np.int64)

In [5]:
lincs_index = np.arange(n_samples, dtype=np.int64)

In [6]:
h5_dst = (
    "./Code/other_models/TranSiGen/data/Meisheng_used_data/"
    "processed_data_id.h5"
)

with h5py.File(h5_dst, "w") as f:
    f.create_dataset("LINCS_index",      data=lincs_index, dtype=np.int64)
    f.create_dataset("canonical_smiles", data=canon_idx,   dtype=np.int64)
    f.create_dataset("cid",              data=cid_raw)     # keep original dtype
    f.create_dataset("sig",              data=sig_raw)

print("Wrote", n_samples, "samples to", h5_dst)

Wrote 836649 samples to /work/users/m/e/meisheng/Dissertation/TranSiGen/data/Meisheng_used_data/processed_data_id_halfway.h5


In [7]:
with h5py.File(h5_dst, "r") as f:
    for k in f:
        print(f"{k:17} shape={f[k].shape}  dtype={f[k].dtype}")

print("\nFirst 3 rows:")
with h5py.File(h5_dst, "r") as f:
    for k in ["LINCS_index", "canonical_smiles", "cid", "sig"]:
        print(k, "→", f[k][:3])

LINCS_index       shape=(836649,)  dtype=int64
canonical_smiles  shape=(836649,)  dtype=int64
cid               shape=(836649,)  dtype=|S8
sig               shape=(836649,)  dtype=|S46

First 3 rows:
LINCS_index → [0 1 2]
canonical_smiles → [0 0 0]
cid → [b'A375' b'A375' b'A375']
sig → [b'REP.A001_A375_24H_X1_B22:B13-2' b'REP.A001_A375_24H_X1_B22:B14-2'
 b'REP.A001_A375_24H_X1_B22:B15-2']


In [8]:
def load_from_HDF(fname):
    """Load data from a HDF5 file to a dictionary."""
    data = dict()
    with h5py.File(fname, 'r') as f:
        for key in f:
            data[key] = np.asarray(f[key])
            if isinstance(data[key][0], np.bytes_):
                data[key] = data[key].astype(str)
    return data

data = load_from_HDF(h5_dst)

In [9]:
data

{'LINCS_index': array([     0,      1,      2, ..., 836646, 836647, 836648]),
 'canonical_smiles': array([   0,    0,    0, ..., 1419, 1419, 1419]),
 'cid': array(['A375', 'A375', 'A375', ..., 'PC3', 'PC3', 'PC3'], dtype='<U8'),
 'sig': array(['REP.A001_A375_24H_X1_B22:B13-2', 'REP.A001_A375_24H_X1_B22:B14-2',
        'REP.A001_A375_24H_X1_B22:B15-2', ...,
        'PCLB003_PC3_24H_X3_B13:P22-1', 'PCLB003_PC3_24H_X3_B13:P23-1',
        'PCLB003_PC3_24H_X3_B13:P24-1'], dtype='<U46')}

In [10]:
np.unique(data["canonical_smiles"]).size

17766

In [11]:
np.unique(data["LINCS_index"]).size

836649

In [12]:
np.unique(data["cid"]).size

82

In [13]:
np.unique(data["sig"]).size

836649